# Datawhale 学习项目｜《Hello-Agents》学习笔记 + 代码实践

> Datawhale 学习项目：基于 Datawhale《Hello-Agents》V1.0.3（2026-07-17 PDF）整理。
> 覆盖范围：**第一章 初识智能体**、**第二章 智能体发展史**、**第三章 大语言模型基础**。

## 使用说明

- 标注“必做代码”的单元格只使用 **Python 标准库**，可直接运行。
- 标注“可选进阶”的单元格需要额外依赖（如 `openai`、`torch`、`transformers`），默认关闭，避免联网和下载大模型。
- 笔记部分按教材结构梳理核心概念；代码部分尽量复现教材案例，并补充少量可运行 demo，方便课堂展示与学习复盘。

## 目录

1. 第一章 初识智能体：定义、构成、运行机制、Thought-Action-Observation、Workflow vs Agent
2. 第二章 智能体发展史：符号主义、ELIZA、心智社会、学习范式、现代 LLM 智能体
3. 第三章 大语言模型基础：N-gram、Transformer、提示工程、分词、缩放法则与幻觉

---

# 第一章 初识智能体

## 1.1 什么是智能体？

教材定义：**智能体（Agent）是能够通过传感器（Sensors）感知环境（Environment），并通过执行器（Actuators）自主采取行动（Action）以达成目标的实体。**

四个基本要素：

| 要素 | 含义 | 例子 |
|---|---|---|
| 环境 Environment | 智能体所处的外部世界 | 道路交通、金融市场、旅行场景 |
| 传感器 Sensors | 获取环境信息的通道 | 摄像头、API 返回数据、用户输入 |
| 执行器 Actuators | 改变环境状态的手段 | 机械臂、代码解释器、搜索 API |
| 自主性 Autonomy | 基于感知和内部状态独立决策 | 不是简单 if-else，而是主动规划 |

### 传统智能体的演进阶梯

1. **简单反射智能体**：条件-动作规则，只看当前感知。例：恒温器。
2. **基于模型的反射智能体**：拥有内部世界模型，能追踪不可直接感知的状态。例：隧道中自动驾驶。
3. **基于目标的智能体**：主动规划以达成某个未来状态。例：GPS 导航。
4. **基于效用的智能体**：在多个目标间权衡，最大化期望效用。
5. **学习型智能体**：通过强化学习等方式，根据环境反馈自我改进。

### LLM 驱动的新范式

传统智能体靠工程师显式编程；LLM 智能体通过海量数据预训练获得隐式世界模型与涌现能力。教材用“智能旅行助手”说明其能力：

- **规划与推理**：把“规划一次厦门之旅”分解为确认偏好 → 查询信息 → 制定行程 → 预订票务住宿。
- **工具使用**：识别信息缺口，主动调用天气、地图、预订 API。
- **动态修正**：把用户反馈（如“超出预算”）视为新约束，重新规划。

### 三种分类维度

- 按内部决策架构：反应式、模型式、目标式、效用式、学习式。
- 按时间与反应性：**反应式**（快、低开销、短视）、**规划式**（深远但慢）、**混合式**（思考-行动-观察循环）。
- 按知识表示：**符号主义**（显式规则，可解释但脆弱）、**亚符号主义**（分布式向量，鲁棒但不透明）、**神经符号主义**（ Neural + Symbolic，LLM Agent 是典型实践）。

## 1.2 构成与运行原理

### PEAS 模型

PEAS = Performance（性能度量）、Environment（环境）、Actuators（执行器）、Sensors（传感器）。它用于精确描述一个任务环境。

教材指出数字环境常具有：**部分可观察、随机性、多智能体、序贯且动态**等特性。

### 智能体循环 Agent Loop

核心闭环：**感知 Perception → 思考 Thought（规划 + 工具选择）→ 行动 Action → 观察 Observation → 再感知**。

工程中常用结构化交互协议：

```text
Thought: 分析当前情境、回顾观察、分解问题、规划下一步
Action:  function_name(arg="value") 或 Finish[最终答案]
Observation: 工具执行结果被整理成的简洁自然语言
```

## 1.3 动手体验：5 分钟实现第一个智能体

教材案例任务：“查询今天北京天气，然后根据天气推荐合适景点。”

需要：指令模板 `AGENT_SYSTEM_PROMPT`、工具 `get_weather` / `get_attraction`、LLM 客户端、主循环解析 `Thought/Action/Observation`。

关键点：

- 工具函数放入字典，主循环根据 Action 字符串调用。
- 用正则解析 `Action`，用 `Finish[...]` 结束任务。
- 设置最大循环次数，防止无限循环。

## 1.4 协作模式与 Workflow 差异

- **作为开发者工具**：GitHub Copilot、Cursor、Claude Code、Trae 等，增强人类工作流。
- **作为自主协作者**：BabyAGI、AutoGPT、CrewAI、AutoGen、MetaGPT、LangGraph 等，接受高层目标自主规划执行。
- **Workflow vs Agent**：Workflow 是预设流程、静态编排；Agent 是目标导向、动态推理。前者可控稳定，后者灵活适应开放任务。

In [1]:
# 必做代码 1.1：用 PEAS 描述“智能旅行助手”的任务环境
travel_peas = {
    "Performance 性能度量": [
        "行程是否满足用户偏好与预算",
        "推荐景点的相关性与可执行性",
        "响应速度与用户满意度",
    ],
    "Environment 环境": [
        "天气、航班/酒店实时数据",
        "地图与景点开放信息",
        "其他用户与自动定价系统",
    ],
    "Actuators 执行器": [
        "天气查询 API",
        "景点搜索 API",
        "票务/酒店预订接口",
    ],
    "Sensors 传感器": [
        "用户自然语言输入",
        "API 返回的 JSON 数据",
        "历史对话与用户偏好记录",
    ],
}

env_properties = {
    "部分可观察": "一次查询无法获得所有航司全部实时座位，只能看到 API 返回的部分数据。",
    "随机性": "相邻两次票价查询可能返回不同价格和余票。",
    "多智能体": "其他用户、自动脚本、航司调价系统都会改变环境状态。",
    "序贯且动态": "当前动作影响未来，且环境在决策过程中持续变化。",
}

for key, value in travel_peas.items():
    print(f"[{key}]")
    for item in value:
        print(f"  - {item}")
print("\n[环境特性]")
for key, value in env_properties.items():
    print(f"  {key}: {value}")

[Performance 性能度量]
  - 行程是否满足用户偏好与预算
  - 推荐景点的相关性与可执行性
  - 响应速度与用户满意度
[Environment 环境]
  - 天气、航班/酒店实时数据
  - 地图与景点开放信息
  - 其他用户与自动定价系统
[Actuators 执行器]
  - 天气查询 API
  - 景点搜索 API
  - 票务/酒店预订接口
[Sensors 传感器]
  - 用户自然语言输入
  - API 返回的 JSON 数据
  - 历史对话与用户偏好记录

[环境特性]
  部分可观察: 一次查询无法获得所有航司全部实时座位，只能看到 API 返回的部分数据。
  随机性: 相邻两次票价查询可能返回不同价格和余票。
  多智能体: 其他用户、自动脚本、航司调价系统都会改变环境状态。
  序贯且动态: 当前动作影响未来，且环境在决策过程中持续变化。


In [2]:
# 必做代码 1.2：最小可运行的 Thought-Action-Observation 旅行助手
# 说明：这里用 MockLLM 模拟教材中真实 LLM 的决策输出，保证无需 API Key 也能运行。
import re

AGENT_SYSTEM_PROMPT = """
你是一个智能旅行助手。每次回复必须严格包含一对 Thought 和 Action。
可用工具：
- get_weather(city: str)
- get_attraction(city: str, weather: str)
Action 格式：function_name(arg="value") 或 Finish[最终答案]
"""

WEATHER_DB = {
    "北京": ("Sunny", 26),
    "上海": ("Rainy", 18),
    "广州": ("Cloudy", 24),
}

ATTRACTION_DB = {
    ("北京", "Sunny"): "颐和园（湖景与古建筑）、长城（壮观景观与历史意义）",
    ("北京", "Rainy"): "首都博物馆、国家博物馆等室内场所",
    ("上海", "Rainy"): "上海博物馆、上海科技馆",
    ("广州", "Cloudy"): "越秀公园、陈家祠",
}


def get_weather(city: str) -> str:
    weather, temp = WEATHER_DB.get(city, ("Sunny", 22))
    return f"{city} 当前天气:{weather}，气温 {temp} 摄氏度"


def get_attraction(city: str, weather: str) -> str:
    return ATTRACTION_DB.get(
        (city, weather),
        f"{city} 在 {weather} 天气下推荐：城市公园、博物馆与本地文化街区。",
    )


available_tools = {
    "get_weather": get_weather,
    "get_attraction": get_attraction,
}


def parse_action(text: str):
    """从 Thought-Action 文本中解析 Action 名称和参数。"""
    m = re.search(r"Action:\s*(.+)", text)
    if not m:
        return None, {}
    action = m.group(1).strip()
    if action.startswith("Finish["):
        answer = action[len("Finish["):].rstrip("]").strip()
        return "Finish", {"answer": answer}
    call = re.match(r"(\w+)\((.*)\)", action)
    if not call:
        return None, {}
    name, args_str = call.group(1), call.group(2)
    kwargs = dict(re.findall(r'(\w+)\s*=\s*"([^"]*)"', args_str))
    return name, kwargs


class MockLLM:
    """根据 prompt 中的观察状态，按教材格式生成下一轮 Thought-Action。"""

    def generate(self, prompt: str, system_prompt: str = None) -> str:
        if "Observation:" not in prompt:
            m = re.search(r"(北京|上海|广州)", prompt)
            city = m.group(1) if m else "北京"
            return (
                "Thought: 首先需要获取天气，再基于天气推荐景点。\n"
                f'Action: get_weather(city="{city}")'
            )

        obs = re.findall(r"Observation:\s*(.+)", prompt)[-1]
        if "当前天气" in obs:
            city = re.split(r"\s*当前天气", obs)[0].strip()
            weather = "Sunny" if "Sunny" in obs else ("Rainy" if "Rainy" in obs else "Cloudy")
            return (
                "Thought: 已获得天气信息，接下来据此选择景点。\n"
                f'Action: get_attraction(city="{city}", weather="{weather}")'
            )

        return (
            "Thought: 已获得景点推荐信息，可以汇总回答用户。\n"
            "Action: Finish[已根据天气完成景点推荐，祝旅途愉快！]"
        )


def run_agent(user_prompt: str, max_steps: int = 5):
    llm = MockLLM()
    history = [f"用户请求: {user_prompt}"]
    print(f"用户输入: {user_prompt}\n" + "=" * 50)

    for step in range(1, max_steps + 1):
        print(f"\n--- 循环 {step} ---")
        full_prompt = "\n".join(history)
        output = llm.generate(full_prompt, AGENT_SYSTEM_PROMPT)
        print(output)
        history.append(output)

        name, kwargs = parse_action(output)
        if name == "Finish":
            print(f"\n任务完成，最终答案: {kwargs['answer']}")
            return kwargs["answer"]

        if name in available_tools:
            observation = available_tools[name](**kwargs)
        else:
            observation = f"错误: 未定义的工具 '{name}'"

        obs_str = f"Observation: {observation}"
        print(obs_str)
        history.append(obs_str)

    print("达到最大循环次数，任务终止。")
    return None


answer = run_agent("你好，请帮我查询一下今天北京的天气，然后根据天气推荐一个合适的旅游景点。")

用户输入: 你好，请帮我查询一下今天北京的天气，然后根据天气推荐一个合适的旅游景点。

--- 循环 1 ---
Thought: 首先需要获取天气，再基于天气推荐景点。
Action: get_weather(city="北京")
Observation: 北京 当前天气:Sunny，气温 26 摄氏度

--- 循环 2 ---
Thought: 已获得天气信息，接下来据此选择景点。
Action: get_attraction(city="北京", weather="Sunny")
Observation: 颐和园（湖景与古建筑）、长城（壮观景观与历史意义）

--- 循环 3 ---
Thought: 已获得景点推荐信息，可以汇总回答用户。
Action: Finish[已根据天气完成景点推荐，祝旅途愉快！]

任务完成，最终答案: 已根据天气完成景点推荐，祝旅途愉快！


In [3]:
# 可选进阶 1.3：把 MockLLM 换成兼容 OpenAI 接口的真实 LLM 客户端
# 默认不执行；如需运行，把下面开关改为 True 并配置环境变量。
RUN_REAL_API = False

if RUN_REAL_API:
    import os
    from openai import OpenAI

    class OpenAICompatibleClient:
        """调用任何兼容 OpenAI 接口规范的 LLM 服务。"""

        def __init__(self, model: str, api_key: str, base_url: str):
            self.model = model
            self.client = OpenAI(api_key=api_key, base_url=base_url)

        def generate(self, prompt: str, system_prompt: str) -> str:
            messages = [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": prompt},
            ]
            response = self.client.chat.completions.create(
                model=self.model, messages=messages, stream=False
            )
            return response.choices[0].message.content

    llm = OpenAICompatibleClient(
        model=os.environ["MODEL_ID"],
        api_key=os.environ["API_KEY"],
        base_url=os.environ["BASE_URL"],
    )
    print("真实 LLM 客户端已就绪。将 run_agent 中的 MockLLM() 替换为 llm 即可。")
else:
    print("未启用真实 API，继续使用 MockLLM 保证作业可离线运行。")

未启用真实 API，继续使用 MockLLM 保证作业可离线运行。


In [4]:
# 必做代码 1.4：Workflow 与 Agent 的差异（以售后退款为例）
def workflow_refund(days: int, amount: float, custom: bool):
    """固定流程：每条判断条件都由开发者预先写死。"""
    if custom:
        return "拒绝", ["规则: 定制品一律拒绝退款"]
    if days > 7:
        return "客服审核", ["规则: 超过 7 天必须人工审核"]
    if amount < 100:
        return "自动通过", ["规则: 7 天内且金额 < 100 自动通过"]
    if amount <= 500:
        return "客服审核", ["规则: 7 天内且 100-500 元由客服审核"]
    return "主管审批", ["规则: 7 天内且 > 500 元由主管审批"]


def agent_refund_decision(order: dict):
    """目标导向：综合多因素推理，可处理规则未明确覆盖的新情况。"""
    score = 0
    reasons = []
    if order["days"] <= 7:
        score += 2
        reasons.append("购买时间在 7 天内，倾向批准")
    else:
        reasons.append("超过 7 天，需要更多证据")
    if not order["custom"]:
        score += 1
        reasons.append("非定制商品，可正常退货")
    else:
        score -= 3
        reasons.append("定制品通常不可退，需谨慎")
    if order["unused"]:
        score += 2
        reasons.append("商品未使用，状态良好")
    if order["vip_years"] >= 3:
        score += 1
        reasons.append("长期用户，历史信用较好")

    decision = "批准" if score >= 3 else ("人工复核" if score >= 0 else "拒绝")
    return decision, reasons


orders = [
    {"days": 3, "amount": 680, "custom": False, "unused": True, "vip_years": 4},
    {"days": 10, "amount": 80, "custom": False, "unused": True, "vip_years": 1},
    {"days": 2, "amount": 200, "custom": True, "unused": False, "vip_years": 5},
]

for i, order in enumerate(orders, 1):
    w_decision, w_reasons = workflow_refund(
        order["days"], order["amount"], order["custom"]
    )
    a_decision, a_reasons = agent_refund_decision(order)
    print(f"订单 {i}:")
    print(f"  Workflow -> {w_decision} | {w_reasons[0]}")
    print(f"  Agent    -> {a_decision} | {a_reasons[0]}")

订单 1:
  Workflow -> 主管审批 | 规则: 7 天内且 > 500 元由主管审批
  Agent    -> 批准 | 购买时间在 7 天内，倾向批准
订单 2:
  Workflow -> 客服审核 | 规则: 超过 7 天必须人工审核
  Agent    -> 批准 | 超过 7 天，需要更多证据
订单 3:
  Workflow -> 拒绝 | 规则: 定制品一律拒绝退款
  Agent    -> 人工复核 | 购买时间在 7 天内，倾向批准


### 第一章小结

- 智能体 = 感知环境 + 自主决策 + 行动 + 反馈循环。
- PEAS 用于规约任务环境；Agent Loop 是运行骨架；Thought/Action/Observation 是工程协议。
- LLM 智能体把语言模型作为“大脑”，通过工具使用把语言推理连接到真实环境。
- 现实系统常采用 Workflow + Agent 的混合方案：稳定流程用 Workflow，开放任务用 Agent。

---

# 第二章 智能体发展史

## 2.1 符号与逻辑的早期智能体

### 物理符号系统假说 PSSH（Newell & Simon, 1976）

- **充分性论断**：任何物理符号系统都具备产生通用智能行为的充分手段。
- **必要性论断**：任何展现通用智能行为的系统，本质上都是物理符号系统。

核心信念：**智能的本质是符号的计算与处理**。

### 专家系统

结构：**知识库 + 推理机 + 用户界面**，体现“知识与推理分离”。

- 知识表示：产生式规则 `IF ... THEN ...`。
- 推理方式：**正向链**（数据驱动）与**反向链**（目标驱动）。
- 代表案例 **MYCIN**：约 600 条规则，反向链诊断细菌感染，引入置信因子 CF 处理不确定性。

### SHRDLU（Winograd, 1968-1970）

在“积木世界”中集成自然语言理解、指代消解、上下文记忆、规划与行动。它是早期“感知-思考-行动”闭环的典范。

### 符号主义的根本挑战

1. **知识获取瓶颈**：专家知识难以大规模手工编码。
2. **常识问题**：人类默认掌握的常识无法穷举。
3. **框架问题**：动作后哪些状态未改变，难以高效显式声明。
4. **系统脆弱性**：规则之外的新情况可能导致系统完全失灵。

## 2.2 ELIZA：基于规则的聊天机器人

ELIZA（Weizenbaum, 1966）通过 **关键词识别 → 分解规则（通配符）→ 代词转换 → 重组规则** 生成看似共情的回应。它并不理解语义，却引发了著名的“ELIZA 效应”。

局限：缺乏语义理解、无上下文记忆、规则扩展性差。

## 2.3 明斯基的“心智社会”

核心观点：智能并非来自单一完美推理器，而是来自大量简单、专门化心智过程的协作与**涌现**。

- 智能体（agent）是简单的“无心”过程，如 LINE-FINDER、GRASP。
- 多个智能体组成机构（agency），如 BUILD-TOWER。
- 复杂行为从局部激活/抑制中涌现，没有全局规划者。
- 对多智能体系统的启发：去中心化控制、涌现式计算、智能体社会性。

## 2.4 学习范式与现代智能体

| 阶段 | 核心思想 | 代表 |
|---|---|---|
| 联结主义 | 知识分布式存储在连接权重中，通过反向传播学习 | 神经网络、深度学习 |
| 强化学习 | 在试错中最大化累积奖励，解决序贯决策 | AlphaGo、TD-Gammon |
| 大规模预训练 | 自监督学习世界知识，再微调下游任务 | GPT、BERT |
| LLM 智能体 | 感知、规划、记忆、工具、执行模块协同 | ChatGPT Agent、AutoGPT 等 |

涌现能力：上下文学习、思维链推理、指令遵循等。

现代 LLM 智能体流程：**感知 → 思考（规划 + LLM + 记忆）→ 行动（执行模块调用工具）→ 观察（工具结果 + 新环境状态）→ 更新记忆并循环**。

In [5]:
# 必做代码 2.1：一个可解释的专家系统（正向链 + 置信因子 CF）
def combine_cf(cf1: float, cf2: float) -> float:
    """MYCIN 风格的置信因子组合：两者共同支持同一结论。"""
    return cf1 + cf2 * (1 - cf1)


rules = [
    {"if": {"fever": True, "cough": True}, "then": "respiratory_infection", "cf": 0.75},
    {"if": {"fever": True, "headache": True}, "then": "respiratory_infection", "cf": 0.45},
    {"if": {"respiratory_infection": 0.7}, "then": "recommend_antibiotic", "cf": 0.80},
]


def condition_matched(condition: dict, state: dict) -> bool:
    for key, expected in condition.items():
        actual = state.get(key)
        if isinstance(expected, float):
            if not isinstance(actual, (int, float)) or actual < expected:
                return False
        elif actual != expected:
            return False
    return True


def forward_chain(initial_facts: dict, rule_set: list):
    state = dict(initial_facts)
    trace = []
    changed = True
    while changed:
        changed = False
        for rule in rule_set:
            if not condition_matched(rule["if"], state):
                continue
            conclusion = rule["then"]
            old = state.get(conclusion, 0.0)
            new_cf = (
                combine_cf(old, rule["cf"]) if isinstance(old, (int, float)) else rule["cf"]
            )
            if conclusion not in state or new_cf > old:
                state[conclusion] = new_cf
                trace.append(f"{conclusion} <- CF {new_cf:.3f} (规则: {rule['if']})")
                changed = True
    return state, trace


facts = {"fever": True, "cough": True, "headache": True}
result, trace = forward_chain(facts, rules)
print("推理轨迹:")
for line in trace:
    print("  " + line)
print("\n最终状态:", {k: (round(v, 3) if isinstance(v, float) else v) for k, v in result.items()})

推理轨迹:
  respiratory_infection <- CF 0.750 (规则: {'fever': True, 'cough': True})
  respiratory_infection <- CF 0.863 (规则: {'fever': True, 'headache': True})
  recommend_antibiotic <- CF 0.800 (规则: {'respiratory_infection': 0.7})
  respiratory_infection <- CF 0.966 (规则: {'fever': True, 'cough': True})
  respiratory_infection <- CF 0.981 (规则: {'fever': True, 'headache': True})
  recommend_antibiotic <- CF 0.960 (规则: {'respiratory_infection': 0.7})
  respiratory_infection <- CF 0.995 (规则: {'fever': True, 'cough': True})
  respiratory_infection <- CF 0.997 (规则: {'fever': True, 'headache': True})
  recommend_antibiotic <- CF 0.992 (规则: {'respiratory_infection': 0.7})
  respiratory_infection <- CF 0.999 (规则: {'fever': True, 'cough': True})
  respiratory_infection <- CF 1.000 (规则: {'fever': True, 'headache': True})
  recommend_antibiotic <- CF 0.998 (规则: {'respiratory_infection': 0.7})
  respiratory_infection <- CF 1.000 (规则: {'fever': True, 'cough': True})
  respiratory_infection <- CF 1.000 (

In [6]:
# 必做代码 2.2：复现教材中的迷你 ELIZA（不阻塞输入，使用预置对话演示）
import re
import random

random.seed(2026)

rules = {
    r"I need (.*)": [
        "Why do you need {0}?",
        "Would it really help you to get {0}?",
        "Are you sure you need {0}?",
    ],
    r"Why don\'t you (.*)\?": [
        "Do you really think I don't {0}?",
        "Perhaps eventually I will {0}.",
        "Do you really want me to {0}?",
    ],
    r"Why can\'t I (.*)\?": [
        "Do you think you should be able to {0}?",
        "If you could {0}, what would you do?",
        "I don't know -- why can't you {0}?",
    ],
    r"I am (.*)": [
        "Did you come to me because you are {0}?",
        "How long have you been {0}?",
        "How do you feel about being {0}?",
    ],
    r".* mother .*": [
        "Tell me more about your mother.",
        "What was your relationship with your mother like?",
    ],
    r".* father .*": [
        "Tell me more about your father.",
        "How did your father make you feel?",
    ],
    r".*": [
        "Please tell me more.",
        "Let's change focus a bit... Tell me about your family.",
        "Can you elaborate on that?",
    ],
}

pronoun_swap = {
    "i": "you", "you": "i", "me": "you", "my": "your",
    "am": "are", "are": "am", "was": "were",
    "i'd": "you would", "i've": "you have", "i'll": "you will",
    "yours": "mine", "mine": "yours",
}


def swap_pronouns(phrase: str) -> str:
    words = phrase.lower().split()
    return " ".join(pronoun_swap.get(word, word) for word in words)


def eliza_respond(user_input: str) -> str:
    for pattern, responses in rules.items():
        match = re.search(pattern, user_input, re.IGNORECASE)
        if match:
            captured = match.group(1) if match.groups() else ""
            transformed = swap_pronouns(captured)
            return random.choice(responses).format(transformed)
    return random.choice(rules[r".*"])


demo_dialogue = [
    "I am feeling sad today.",
    "I need some help with my project.",
    "My mother is not happy with my work.",
    "Why can't I sleep well?",
]

print("Therapist: Hello! How can I help you today?")
for utterance in demo_dialogue:
    print(f"You: {utterance}")
    print(f"Therapist: {eliza_respond(utterance)}")

Therapist: Hello! How can I help you today?
You: I am feeling sad today.
Therapist: Did you come to me because you are feeling sad today.?
You: I need some help with my project.
Therapist: Would it really help you to get some help with your project.?
You: My mother is not happy with my work.
Therapist: Tell me more about your mother.
You: Why can't I sleep well?
Therapist: Do you think you should be able to sleep well?


In [7]:
# 必做代码 2.3：“心智社会”玩具模型——多个简单智能体协作搭积木塔
# 没有任何单一智能体知道完整计划；每个 agent 只看局部状态并做一步动作。
state = {
    "target_order": ["blue", "red", "green"],
    "next_index": 0,
    "hand": None,
    "target": None,
    "tower": [],
}


def agent_find_block(s):
    if s["hand"] is None and s["target"] is None and s["next_index"] < len(s["target_order"]):
        block = s["target_order"][s["next_index"]]
        s["target"] = block
        return f"FIND-BLOCK: 锁定目标 {block}"


def agent_reach_grasp(s):
    if s["target"] and s["hand"] is None:
        s["hand"] = s["target"]
        return f"REACH+GRASP: 抓起 {s['hand']}"


def agent_put_on_top(s):
    if s["hand"] is not None and s["target"] is not None:
        block = s["hand"]
        s["tower"].append(block)
        s["hand"] = None
        s["target"] = None
        s["next_index"] += 1
        return f"PUT-ON-TOP: 放置 {block}，当前塔 = {s['tower']}"


def agent_stop(s):
    if len(s["tower"]) == len(s["target_order"]):
        return "STOP: 塔已搭完"


simple_agents = [agent_find_block, agent_reach_grasp, agent_put_on_top, agent_stop]

for step in range(20):
    activated = False
    for agent in simple_agents:
        message = agent(state)
        if message:
            print(f"step {step + 1:02d} | {message}")
            activated = True
            break
    if not activated or state["tower"] == state["target_order"]:
        break

print("\n最终塔:", state["tower"])

step 01 | FIND-BLOCK: 锁定目标 blue
step 02 | REACH+GRASP: 抓起 blue
step 03 | PUT-ON-TOP: 放置 blue，当前塔 = ['blue']
step 04 | FIND-BLOCK: 锁定目标 red
step 05 | REACH+GRASP: 抓起 red
step 06 | PUT-ON-TOP: 放置 red，当前塔 = ['blue', 'red']
step 07 | FIND-BLOCK: 锁定目标 green
step 08 | REACH+GRASP: 抓起 green
step 09 | PUT-ON-TOP: 放置 green，当前塔 = ['blue', 'red', 'green']

最终塔: ['blue', 'red', 'green']


In [8]:
# 必做代码 2.4：Q-learning 网格世界——强化学习的“试错”循环
import random

random.seed(7)
N = 4
START = (0, 0)
GOAL = (3, 3)
OBSTACLES = {(1, 1), (2, 2)}
ACTIONS = [(1, 0), (-1, 0), (0, 1), (0, -1)]  # 下、上、右、左


def env_step(state, action):
    nr, nc = state[0] + action[0], state[1] + action[1]
    if not (0 <= nr < N and 0 <= nc < N) or (nr, nc) in OBSTACLES:
        return state, -0.2, False
    next_state = (nr, nc)
    if next_state == GOAL:
        return next_state, 10.0, True
    return next_state, -0.05, False


Q = {}


def q_value(state, action):
    return Q.get((state, action), 0.0)


alpha, gamma, epsilon = 0.2, 0.9, 0.2
for episode in range(400):
    s = START
    for _ in range(60):
        if random.random() < epsilon:
            a = random.randrange(4)
        else:
            a = max(range(4), key=lambda i: q_value(s, i))
        s2, reward, done = env_step(s, ACTIONS[a])
        best_next = max(q_value(s2, i) for i in range(4))
        Q[(s, a)] = q_value(s, a) + alpha * (
            reward + gamma * best_next - q_value(s, a)
        )
        s = s2
        if done:
            break

# 用学到的策略走一遍
path = [START]
s = START
visited = {START}
for _ in range(20):
    if s == GOAL:
        break
    a = max(range(4), key=lambda i: q_value(s, i))
    s2, _, _ = env_step(s, ACTIONS[a])
    if s2 in visited:
        break
    path.append(s2)
    visited.add(s2)
    s = s2

print("训练回合数: 400")
print("贪心策略路径:", path)
print("是否到达终点:", path[-1] == GOAL)

训练回合数: 400
贪心策略路径: [(0, 0), (0, 1), (0, 2), (1, 2), (1, 3), (2, 3), (3, 3)]
是否到达终点: True


### 第二章小结

- 符号主义用“知识 + 推理”模拟智能，成功于封闭领域，但受限于知识获取瓶颈和脆弱性。
- ELIZA 证明简单规则也能制造“智能错觉”，但没有真正理解。
- 心智社会把智能看作大量简单过程的涌现，为多智能体系统奠定思想基础。
- 联结主义、强化学习、预训练与 LLM 逐步融合，形成今天的 Agent 架构。

---

# 第三章 大语言模型基础

## 3.1 语言模型与 Transformer 架构

### 从 N-gram 到 RNN

语言模型的根本任务：计算词序列概率。

链式法则：

```text
P(w1, w2, ..., wm) = P(w1) · P(w2|w1) · P(w3|w1,w2) · ... · P(wm|w1,...,w_{m-1})
```

马尔可夫假设：一个词的概率只与前面有限个词有关。由此得到 N-gram：

- Bigram：`P(wi|w_{i-1})`
- Trigram：`P(wi|w_{i-2}, w_{i-1})`

概率用最大似然估计（计数）得到。局限：**数据稀疏**与**泛化能力差**。

神经网络语言模型用**词嵌入**把词映射为连续向量，语义相近的词向量相近。RNN 引入隐藏状态作为短期记忆；LSTM 用遗忘门、输入门、输出门缓解长期依赖问题。

### Transformer

2017 年《Attention Is All You Need》提出 Transformer，完全抛弃循环结构，靠注意力机制实现并行计算。

- Encoder：理解整个输入，为每个 token 生成上下文向量。
- Decoder：参考已生成前文并“咨询”编码器，生成下一个词。
- Self-Attention：每个词用 Query 去查询所有词的 Key，并用softmax权重对 Value 加权求和。

```text
Attention(Q, K, V) = softmax(QK^T / sqrt(d_k)) V
```

- Multi-Head Attention：把 Q/K/V 切成 h 份，并行从不同子空间学习关系。
- FFN：逐位置前馈网络，先升维再降维。
- Add & Norm：残差连接 + 层归一化，稳定深层训练。
- Positional Encoding：用正弦/余弦函数给 token 注入顺序信息。

### Decoder-Only 与自回归生成

GPT 只保留 Decoder，把任务统一为“预测下一个词”。训练时用**因果掩码**防止看到未来 token；生成时把已生成内容不断拼回输入，自回归地继续预测。

优势：训练目标统一、结构简单易扩展、天然适合生成任务。

## 3.2 与大语言模型交互

### 采样参数

- **Temperature**：T 小 → 分布更陡，输出确定；T 大 → 分布更平，输出多样。
- **Top-k**：只保留概率最高的 k 个 token。
- **Top-p（核采样）**：保留累积概率首次达到 p 的最小集合。
- 同时设置时通常按 **Temperature → Top-k → Top-p** 分层过滤；实际候选取交集。

### 提示策略

- Zero-shot：不给示例，直接下指令。
- One-shot：给一个示例。
- Few-shot：给多个示例，帮助模型理解任务边界。
- 指令调优让模型能直接遵循自然语言指令。
- 角色扮演、上下文示例、思维链（CoT）是常用技巧。

### 文本分词

现代 LLM 普遍用子词分词：

- Word-based：词表大、OOV 问题。
- Character-based：无 OOV，但语义弱、序列长。
- Subword：常见词保留，罕见词拆成子词。
- **BPE**：迭代合并最高频相邻 token 对。WordPiece 用于 BERT；SentencePiece 用于 Llama。

分词影响上下文窗口、API 成本和模型表现。

### 模型选择与本地部署

选型维度：性能、成本、延迟、上下文窗口、部署方式、生态工具链、可微调性、安全性。闭源模型开箱即用；开源模型可控、可定制。

## 3.3 缩放法则与局限

- 缩放法则：Loss 与参数量、数据量、计算量呈可预测的幂律关系。
- Chinchilla 定律：给定算力时，参数量与训练数据量应有最优配比，不能只追求参数量。
- 涌现能力：规模跨过阈值后出现 CoT、指令遵循、代码生成等新能力。
- 幻觉类型：事实性幻觉、忠实性幻觉、内在幻觉。
- 缓解方法：数据清洗与 RLHF、表达不确定性、RAG、多步推理与验证、外部工具调用。

In [9]:
# 必做代码 3.1：Bigram + 最大似然估计（复现教材迷你语料）
import collections

corpus = "datawhale agent learns datawhale agent works"
tokens = corpus.split()
total_tokens = len(tokens)
bigram_counts = collections.Counter(zip(tokens, tokens[1:]))


def bigram_sentence_prob(sentence: str, unigram_counts, pair_counts, total: int):
    words = sentence.split()
    prob = unigram_counts[words[0]] / total
    details = [f"P({words[0]}) = {unigram_counts[words[0]]}/{total} = {prob:.3f}"]
    for prev, cur in zip(words, words[1:]):
        pair_prob = pair_counts[(prev, cur)] / unigram_counts[prev]
        prob *= pair_prob
        details.append(
            f"P({cur}|{prev}) = {pair_counts[(prev, cur)]}/{unigram_counts[prev]} = {pair_prob:.3f}"
        )
    return prob, details


unigram_counts = collections.Counter(tokens)
for sentence in ["datawhale agent learns", "agent works"]:
    prob, details = bigram_sentence_prob(
        sentence, unigram_counts, bigram_counts, total_tokens
    )
    print(f"句子: {sentence}")
    for line in details:
        print("  " + line)
    print(f"  P('{sentence}') ≈ {prob:.3f}\n")

句子: datawhale agent learns
  P(datawhale) = 2/6 = 0.333
  P(agent|datawhale) = 2/2 = 1.000
  P(learns|agent) = 1/2 = 0.500
  P('datawhale agent learns') ≈ 0.167

句子: agent works
  P(agent) = 2/6 = 0.333
  P(works|agent) = 1/2 = 0.500
  P('agent works') ≈ 0.167



In [10]:
# 必做代码 3.2：词嵌入与余弦相似度（教材 king-man+woman 案例）
import math

embeddings = {
    "king": [0.9, 0.8],
    "queen": [0.9, 0.2],
    "man": [0.7, 0.9],
    "woman": [0.7, 0.3],
}


def dot(a, b):
    return sum(x * y for x, y in zip(a, b))


def norm(v):
    return math.sqrt(dot(v, v))


def cosine_similarity(a, b):
    return dot(a, b) / (norm(a) * norm(b))


result_vec = [
    embeddings["king"][i] - embeddings["man"][i] + embeddings["woman"][i]
    for i in range(2)
]
print("king - man + woman =", result_vec)
print("与 queen 的余弦相似度 =", round(cosine_similarity(result_vec, embeddings["queen"]), 4))

print("\n词向量两两相似度:")
words = list(embeddings)
for i, w1 in enumerate(words):
    for w2 in words[i + 1:]:
        print(f"  {w1:6s} vs {w2:6s}: {cosine_similarity(embeddings[w1], embeddings[w2]):.4f}")

king - man + woman = [0.9, 0.2]
与 queen 的余弦相似度 = 1.0

词向量两两相似度:
  king   vs queen : 0.8737
  king   vs man   : 0.9833
  king   vs woman : 0.9487
  queen  vs man   : 0.7706
  queen  vs woman : 0.9827
  man    vs woman : 0.8752


In [11]:
# 必做代码 3.3：手写 Scaled Dot-Product Attention 与因果掩码
import math
import random

random.seed(3)
TOKENS = ["The", "agent", "learns", "because", "it", "is", "intelligent"]
D = 4

token_emb = {t: [random.uniform(-1, 1) for _ in range(D)] for t in TOKENS}
W_q = [[random.uniform(-0.5, 0.5) for _ in range(D)] for _ in range(D)]
W_k = [[random.uniform(-0.5, 0.5) for _ in range(D)] for _ in range(D)]
W_v = [[random.uniform(-0.5, 0.5) for _ in range(D)] for _ in range(D)]


def matvec(W, x):
    return [sum(W[i][j] * x[j] for j in range(len(x))) for i in range(len(W))]


def dot(a, b):
    return sum(x * y for x, y in zip(a, b))


def softmax(xs):
    m = max(xs)
    exps = [math.exp(x - m) for x in xs]
    total = sum(exps)
    return [e / total for e in exps]


def self_attention(causal=False):
    weights, outputs = {}, {}
    scale = math.sqrt(D)
    for i, qi in enumerate(TOKENS):
        q = matvec(W_q, token_emb[qi])
        scores = []
        for j, kj in enumerate(TOKENS):
            if causal and j > i:
                scores.append(-1e9)
            else:
                k = matvec(W_k, token_emb[kj])
                scores.append(dot(q, k) / scale)
        attn_w = softmax(scores)
        out = [0.0] * D
        for j, w in enumerate(attn_w):
            v = matvec(W_v, token_emb[TOKENS[j]])
            for d in range(D):
                out[d] += w * v[d]
        weights[qi], outputs[qi] = attn_w, out
    return weights, outputs


for mode in [False, True]:
    weights, outputs = self_attention(causal=mode)
    title = "因果掩码开启" if mode else "普通自注意力"
    print(f"\n[{title}] token 'it' 的注意力分布:")
    pairs = sorted(zip(TOKENS, weights["it"]), key=lambda x: -x[1])
    for token, weight in pairs:
        print(f"  {token:12s}: {weight:.4f}")
    print(f"  'it' 的输出向量（前2维）: {[round(x, 3) for x in outputs['it'][:2]]}")

print("\n说明: 开启因果掩码后，'it' 只能关注自身及之前的 token，未来位置权重接近 0。")


[普通自注意力] token 'it' 的注意力分布:
  agent       : 0.1649
  The         : 0.1483
  it          : 0.1445
  is          : 0.1432
  intelligent : 0.1416
  learns      : 0.1385
  because     : 0.1190
  'it' 的输出向量（前2维）: [0.168, 0.279]

[因果掩码开启] token 'it' 的注意力分布:
  agent       : 0.2306
  The         : 0.2074
  it          : 0.2021
  learns      : 0.1936
  because     : 0.1663
  is          : 0.0000
  intelligent : 0.0000
  'it' 的输出向量（前2维）: [0.049, 0.13]

说明: 开启因果掩码后，'it' 只能关注自身及之前的 token，未来位置权重接近 0。


In [12]:
# 必做代码 3.4：温度、Top-k、Top-p 对采样分布的影响
import math
import random

base_logits = {"猫": 2.0, "狗": 1.0, "汽车": 0.1, "飞机": -0.5, "鲸鱼": -1.2}


def softmax(xs):
    m = max(xs)
    exps = [math.exp(x - m) for x in xs]
    total = sum(exps)
    return [e / total for e in exps]


def softmax_with_temperature(logits, temperature):
    if temperature <= 0:
        raise ValueError("temperature 必须 > 0")
    scaled = [v / temperature for v in logits.values()]
    probs = softmax(scaled)
    return dict(zip(logits.keys(), probs))


def top_k_filter(probs, k):
    items = sorted(probs.items(), key=lambda kv: -kv[1])[:k]
    total = sum(v for _, v in items)
    return {key: v / total for key, v in items}


def top_p_filter(probs, p):
    items = sorted(probs.items(), key=lambda kv: -kv[1])
    kept, acc = [], 0.0
    for key, value in items:
        kept.append((key, value))
        acc += value
        if acc >= p:
            break
    total = sum(v for _, v in kept)
    return {key: v / total for key, v in kept}


def sample_token(probs, seed=0):
    r = random.Random(seed).random()
    acc = 0.0
    for token, p in probs.items():
        acc += p
        if r <= acc:
            return token
    return list(probs)[-1]


configs = [
    ("低温 T=0.3", dict(temperature=0.3)),
    ("中温 T=0.8", dict(temperature=0.8)),
    ("高温 T=1.5", dict(temperature=1.5)),
    ("Top-k=2", dict(temperature=1.0, k=2)),
    ("Top-p=0.7", dict(temperature=1.0, p=0.7)),
]

for name, cfg in configs:
    probs = softmax_with_temperature(base_logits, cfg["temperature"])
    if "k" in cfg:
        probs = top_k_filter(probs, cfg["k"])
    if "p" in cfg:
        probs = top_p_filter(probs, cfg["p"])
    printed = {k: round(v, 3) for k, v in probs.items()}
    print(f"{name:12s} -> {printed} | 采样结果: {sample_token(probs, seed=5)}")

低温 T=0.3     -> {'猫': 0.964, '狗': 0.034, '汽车': 0.002, '飞机': 0.0, '鲸鱼': 0.0} | 采样结果: 猫
中温 T=0.8     -> {'猫': 0.694, '狗': 0.199, '汽车': 0.065, '飞机': 0.03, '鲸鱼': 0.013} | 采样结果: 猫
高温 T=1.5     -> {'猫': 0.476, '狗': 0.244, '汽车': 0.134, '飞机': 0.09, '鲸鱼': 0.056} | 采样结果: 狗
Top-k=2      -> {'猫': 0.731, '狗': 0.269} | 采样结果: 猫
Top-p=0.7    -> {'猫': 0.731, '狗': 0.269} | 采样结果: 猫


In [13]:
# 必做代码 3.5：Zero-shot / Few-shot / Chain-of-Thought 提示构造
def zero_shot_prompt(text):
    return f"请判断下面文本的情感，只输出正面/负面/中性。\n文本: {text}\n情感:"


def few_shot_prompt(text, examples):
    lines = ["请判断文本情感，只输出正面/负面/中性。"]
    for ex_text, ex_label in examples:
        lines.append(f"文本: {ex_text}\n情感: {ex_label}")
    lines.append(f"文本: {text}\n情感:")
    return "\n".join(lines)


def cot_prompt(question):
    return f"{question}\n请一步一步地思考并解答，最后给出明确答案。"


text = "Datawhale 的 AI Agent 课程非常棒！"
examples = [
    ("这家餐厅的服务太慢了。", "负面"),
    ("这部电影的情节很平淡。", "中性"),
]

print("=== Zero-shot ===")
print(zero_shot_prompt(text))
print("\n=== Few-shot ===")
print(few_shot_prompt(text, examples))
print("\n=== Chain-of-Thought ===")
print(cot_prompt(
    "一个篮球队本赛季 80 场比赛赢了 60%，接下来 15 场赢了 12 场，两个赛季总胜率是多少？"
))

=== Zero-shot ===
请判断下面文本的情感，只输出正面/负面/中性。
文本: Datawhale 的 AI Agent 课程非常棒！
情感:

=== Few-shot ===
请判断文本情感，只输出正面/负面/中性。
文本: 这家餐厅的服务太慢了。
情感: 负面
文本: 这部电影的情节很平淡。
情感: 中性
文本: Datawhale 的 AI Agent 课程非常棒！
情感:

=== Chain-of-Thought ===
一个篮球队本赛季 80 场比赛赢了 60%，接下来 15 场赢了 12 场，两个赛季总胜率是多少？
请一步一步地思考并解答，最后给出明确答案。


In [14]:
# 必做代码 3.6：复现教材 BPE 合并过程，并尝试切分未见词 "bug"
import re
import collections


def get_stats(vocab):
    pairs = collections.defaultdict(int)
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pairs[(symbols[i], symbols[i + 1])] += freq
    return pairs


def merge_vocab(pair, v_in):
    v_out = {}
    bigram = re.escape(" ".join(pair))
    pattern = re.compile(r"(?<!\S)" + bigram + r"(?!\S)")
    for word, freq in v_in.items():
        v_out[pattern.sub("".join(pair), word)] = freq
    return v_out


def learn_bpe(vocab, num_merges):
    merges = []
    for _ in range(num_merges):
        pairs = get_stats(vocab)
        if not pairs:
            break
        best = max(pairs, key=lambda p: pairs[p])
        vocab = merge_vocab(best, vocab)
        merges.append(best)
    return vocab, merges


def apply_bpe(word, merges):
    symbols = list(word) + ["</w>"]
    while True:
        best_pair = None
        best_rank = None
        for i in range(len(symbols) - 1):
            pair = (symbols[i], symbols[i + 1])
            if pair in merges:
                rank = merges.index(pair)
                if best_rank is None or rank < best_rank:
                    best_pair, best_rank = pair, rank
        if best_pair is None:
            break
        new_symbols = []
        i = 0
        while i < len(symbols):
            if i < len(symbols) - 1 and (symbols[i], symbols[i + 1]) == best_pair:
                new_symbols.append("".join(best_pair))
                i += 2
            else:
                new_symbols.append(symbols[i])
                i += 1
        symbols = new_symbols
    return [s.replace("</w>", "") for s in symbols if s.replace("</w>", "")]


vocab = {"h u g </w>": 1, "p u g </w>": 1, "p u n </w>": 1, "b u n </w>": 1}
vocab_after, merges = learn_bpe(vocab, num_merges=4)
print("学习到的合并规则:")
for i, pair in enumerate(merges, 1):
    print(f"  第 {i} 次: {pair} -> {''.join(pair)}")
print("\n最终词表:", list(vocab_after.keys()))
print("未见词 'bug' 的分词结果:", apply_bpe("bug", merges))

学习到的合并规则:
  第 1 次: ('u', 'g') -> ug
  第 2 次: ('ug', '</w>') -> ug</w>
  第 3 次: ('u', 'n') -> un
  第 4 次: ('un', '</w>') -> un</w>

最终词表: ['h ug</w>', 'p ug</w>', 'p un</w>', 'b un</w>']
未见词 'bug' 的分词结果: ['b', 'ug']


In [15]:
# 必做代码 3.7：RAG 思想的最小演示——先检索证据，再构造有依据的回答提示
DOCS = [
    {"id": "D1", "text": "颐和园位于北京海淀，晴天适合泛舟昆明湖并欣赏皇家园林建筑。"},
    {"id": "D2", "text": "首都博物馆与国家博物馆位于北京，雨天适合作为室内游览选择。"},
    {"id": "D3", "text": "八达岭长城是北京著名 historical 景点，晴天气候条件下视野更佳。"},
]
KEYWORDS = ["颐和园", "长城", "博物馆", "雨天", "晴天", "北京"]


def retrieve(query, docs, top_k=2):
    keys = [k for k in KEYWORDS if k in query]
    scored = []
    for doc in docs:
        score = sum(1 for k in keys if k in doc["text"])
        scored.append((score, doc))
    scored.sort(key=lambda x: -x[0])
    return [doc for score, doc in scored[:top_k] if score > 0]


def build_grounded_prompt(query):
    evidence_docs = retrieve(query, DOCS)
    if not evidence_docs:
        return "知识库中没有相关证据。请回答：我不知道，并建议用户补充信息。"
    context = "\n".join(f"[{d['id']}] {d['text']}" for d in evidence_docs)
    return (
        "请只根据以下检索资料回答问题，不要编造资料之外的信息。\n\n"
        f"资料:\n{context}\n\n"
        f"问题: {query}\n回答:"
    )


for query in ["北京晴天去哪玩？", "雨天在北京有什么室内选择？", "火星上有城市吗？"]:
    print("=" * 55)
    print("问题:", query)
    print(build_grounded_prompt(query))

问题: 北京晴天去哪玩？
请只根据以下检索资料回答问题，不要编造资料之外的信息。

资料:
[D1] 颐和园位于北京海淀，晴天适合泛舟昆明湖并欣赏皇家园林建筑。
[D3] 八达岭长城是北京著名 historical 景点，晴天气候条件下视野更佳。

问题: 北京晴天去哪玩？
回答:
问题: 雨天在北京有什么室内选择？
请只根据以下检索资料回答问题，不要编造资料之外的信息。

资料:
[D2] 首都博物馆与国家博物馆位于北京，雨天适合作为室内游览选择。
[D1] 颐和园位于北京海淀，晴天适合泛舟昆明湖并欣赏皇家园林建筑。

问题: 雨天在北京有什么室内选择？
回答:
问题: 火星上有城市吗？
知识库中没有相关证据。请回答：我不知道，并建议用户补充信息。


In [ ]:
# 可选进阶 3.8：本地加载开源 LLM（默认不执行，避免下载大模型）
RUN_LOCAL_LLM = False

if RUN_LOCAL_LLM:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    model_id = "Qwen/Qwen1.5-0.5B-Chat"
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(model_id).to(device)

    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "你好，请介绍一下你自己。"},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer([text], return_tensors="pt").to(device)
    generated = model.generate(inputs.input_ids, max_new_tokens=128)
    generated = [
        out[len(inp):] for inp, out in zip(inputs.input_ids, generated)
    ]
    response = tokenizer.batch_decode(generated, skip_special_tokens=True)[0]
    print("模型回答:", response)
else:
    print("未启用本地模型加载。教材代码已保留，可在网络与算力充足时解除开关。")

未启用本地模型加载。教材代码已保留，可在网络与算力充足时解除开关。


---

# 学习小结

## 本 Notebook 覆盖的作业要求

- **笔记**：前三章核心概念、历史脉络、公式与架构说明，按教材章节整理。
- **代码**：每一章都提供可运行 demo；第一章复现 Agent 循环，第二章复现 ELIZA 与学习范式，第三章复现 N-gram / Attention / BPE / 提示策略 / RAG。
- **可扩展性**：真实 API 与本地开源模型代码以开关形式保留，不影响离线运行。

## 思考题（可作为提交时的自测）

1. 为什么说“Workflow 可控但不够灵活，Agent 灵活但不够可控”？请结合退款案例说明。

2. 符号主义的“知识获取瓶颈”与 LLM 的“幻觉”是否本质上是同一类问题？为什么？
3. 如果让你为论文阅读智能体选择基座模型，你会优先考虑哪些指标？如何缓解长文档与幻觉问题？

## 主要参考

- 第一章参考：Russell & Norvig, *Artificial Intelligence: A Modern Approach*；Kahneman, *Thinking, Fast and Slow*。
- 第二章参考：Newell & Simon (1976)；Weizenbaum (1966)；Minsky (1986)；Silver et al. (2016)。
- 第三章参考：Bengio et al. (2003)；Vaswani et al. (2017)；Radford et al. (2018)；Gage (1994)。

# 思考题参考作答

## 1. 为什么说 Workflow 可控但不灵活，Agent 灵活但不够可控？

**可控性来自“预先写死”。** Workflow 把判断条件、分支顺序和输出结果都提前编码。以退款为例：定制品一律拒绝；7 天内且金额 <100 自动通过；100-500 元客服审核；>500 元主管审批。任何输入都会落到确定分支，过程可预测、可审计、易测试，也便于合规追责。这正是它的优势。

**不灵活也来自“预先写死”。** 它只能处理规则覆盖到的情况。比如“购买第 8 天、商品未拆封、用户是 5 年 VIP”“定制品但商家发错货”这类边界情况，Workflow 只能按硬规则拒绝或转人工，不会综合上下文权衡。规则越加越多还会产生组合爆炸和规则冲突，维护成本迅速上升。

**Agent 的灵活性来自“目标导向 + 上下文推理”。** 在 notebook 的退款示例中，订单 2 超过 7 天，Workflow 只能“客服审核”；Agent 则综合“未使用、长期用户、历史信用较好”等因素给出“批准”。订单 3 是定制品，Workflow 一律拒绝；Agent 考虑到未使用和长期 VIP，给出“人工复核”而不是简单拒绝。Agent 能把政策当作约束，把新情况当作上下文来处理。

**但 Agent 不够可控。** 它的决策来自模型概率推理，会受提示词、采样参数、上下文顺序影响；同样输入未必每次完全一致；还可能产生幻觉或误解政策。它难以像 Workflow 那样给出确定的审计轨迹，延迟和成本也更高。

**结论：实际系统通常采用混合方案。** 标准化、高频、高风险场景用 Workflow 兜底；长尾、模糊、需要权衡的场景交给 Agent，并加上硬规则护栏、日志、人工复核和置信度阈值。这也是第一章“混合式智能体”和“Workflow + Agent”思想的落地。

## 2. 符号主义的“知识获取瓶颈”和 LLM 的“幻觉”是同一类问题吗？

**不是同一类问题，但都属于“知识边界与可靠性”问题，处在智能体系统的不同层面。**

| 维度 | 知识获取瓶颈 | 模型幻觉 |
|---|---|---|
| 出现范式 | 符号主义 / 专家系统 | 大语言模型 / 生成式 AI |
| 根因 | 知识必须靠专家手工编码成 IF-THEN 规则，成本高、难覆盖常识 | 模型按概率自回归预测下一个 token，没有内置事实核查，知识可能缺失、过时或推理链出错 |
| 本质 | **知识获取与表示**问题：知识进不来、表示太刚性 | **生成与验证**问题：知识不一定可靠，却仍被流畅地表达出来 |
| 典型表现 | 遇到规则外情况直接失灵或答“不知道” | 面对不知道的问题也会“自信编造”，看似合理实则错误 |

关键区别在于：**符号系统因为“没规则”而失败，LLM 因为“总想续写”而虚构。** 前者是覆盖不足，后者是误报/虚构。

两者也有联系：它们都说明系统无法保证知识完备且可验证。LLM 通过预训练大幅缓解了“手工编码全部常识”的知识获取瓶颈，但它把风险转移到了新的地方——生成内容缺乏事实锚点，于是产生了幻觉。因此现代智能体常用“神经 + 符号”的混合思路：LLM 负责语言理解、规划和生成；外部知识库、搜索引擎、计算器、代码解释器负责提供事实；规则校验器和引用溯源负责检查结果。RAG、多步验证、工具调用本质上就是给概率模型补上“可检查的证据链”。

## 3. 论文阅读智能体如何选基座模型？如何处理长文档与幻觉？

### 基座模型优先考虑的指标

1. **长上下文与长文本理解能力**：论文动辄几十页，模型不仅要“装得下”，还要能在长文本中准确定位。
2. **推理与公式/表格理解能力**：需要理解研究方法、实验设计、数学推导和结果表格。
3. **指令遵循与结构化输出能力**：能按指定 JSON schema 提取“问题/方法/数据集/结论/局限”，并给出页码或章节引用。
4. **事实忠实度与低幻觉倾向**：回答必须能回溯原文，不能编造论文里没有的内容。
5. **多语言与学术术语理解**：中英文论文混读、专业术语和缩写要准确。
6. **成本、延迟与部署方式**：批量初筛可用轻量模型，深度问答用强模型；涉及未发表稿件时要考虑本地部署与隐私。
7. **生态与工具链**：能否方便接入 RAG、向量库、LangChain/LlamaIndex、PDF 解析与引用定位工具。

实践上常采用分层架构：**强闭源模型（如 GPT-4o / Claude Sonnet 等）负责深度问答与综述，开源模型（如 Qwen / Llama 系列）负责本地化、批量摘要或隐私场景。**

### 长文档处理

- **分块 + RAG**：按章节/段落切分论文，保留章节标题、页码、图表编号等元数据；检索时只把相关块拼入上下文。
- **结构感知切分**：按标题、摘要、方法、实验、图表、参考文献切分，避免把公式和表格从中间截断。
- **层次化摘要 / Map-Reduce**：先逐段生成摘要，再汇总成全文摘要；多篇论文先生成结构化“论文卡片”，再做横向比较。
- **重排序**：检索后对候选片段重新排序，把最相关的证据放在最前面。
- **长上下文模型 + 滑动窗口**：作为补充手段，而不是唯一手段；上下文长不等于能有效利用。

### 幻觉缓解

- **强制基于证据回答**：提示中要求“只根据检索到的原文回答，并给出页码/段落；无证据则回答‘原文未提及’”。
- **多步验证**：先抽取证据 → 再生成答案 → 最后逐条检查答案中的每个断言是否能回溯到证据。
- **外部工具核对**：用计算器/代码执行核对公式和表格数值，用 DOI/arXiv 元数据核对文献信息。
- **程序化校验**：用 JSON schema 约束输出格式，程序检查引用编号是否真实存在。
- **低温度与人在回路**：事实性任务把 temperature 设为 0~0.3；关键结论保留原文片段供用户点击核对，重要场景加入人工复核。

这样设计后，智能体既利用了 LLM 的语言与推理能力，又通过外部证据和校验机制把回答“钉”在原文上。